# Used Cars Price Prediction
Pamela Vilchez | Applied graduate coursework and independent portfolio extension

The coursework source explored vehicle pricing and selected features but did not fit a model. This new notebook rebuilds the analysis from the supplied dataset and adds **Portfolio Extension: Regression Modeling**. It is not a recovered original submission.

Question: how well do vehicle characteristics predict recorded prices in a held-out sample? This is a learning case, not a production valuation service. Price units are retained as supplied; currency and collection date are not established.

Place an authorized copy of `Used Cars-1.csv` in `data/`. The raw dataset is excluded from this repository pending redistribution review. Run from this project directory with the pinned dependencies in requirements.txt.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

SEED = 42
REFERENCE_YEAR = 2019  # Fixed analysis reference, not an asserted collection date.
figures = Path('figures')
figures.mkdir(exist_ok=True)
raw = pd.read_csv('data/Used Cars-1.csv')
expected = ['Year', 'Kilometers_Driven', 'Fuel_Type', 'Transmission', 'Owner_Type',
            'Mileage', 'Engine', 'Power', 'Seats', 'Price']
assert list(raw.columns) == expected, 'Unexpected data schema'
print('Original shape:', raw.shape)
print('Missing values:', raw.isna().sum().to_dict())
print('Exact duplicate rows:', int(raw.duplicated().sum()))

Original shape: (6019, 10)
Missing values: {'Year': 0, 'Kilometers_Driven': 0, 'Fuel_Type': 0, 'Transmission': 0, 'Owner_Type': 0, 'Mileage': 2, 'Engine': 36, 'Power': 143, 'Seats': 42, 'Price': 0}
Exact duplicate rows: 2


## Data cleaning and feature engineering
Remove exact duplicate rows before splitting so duplicate observations cannot straddle training and test sets. This assumes exact duplicates are redundant records; without vehicle IDs we cannot verify identity. Convert numeric fields explicitly. Exclude missing/nonpositive targets. Treat nonpositive mileage, engine, power and seats as missing rather than valid specifications; impute using training data only. Preserve extreme positive values instead of deleting observations to improve scores.

`Car_Age = 2019 - Year` uses a fixed reference year for reproducibility. It is not vehicle age today. Year is then omitted to avoid redundant predictors.

In [2]:
data = raw.drop_duplicates().copy()
numeric_source = ['Year', 'Kilometers_Driven', 'Mileage', 'Engine', 'Power', 'Seats', 'Price']
for column in numeric_source:
    data[column] = pd.to_numeric(data[column], errors='coerce')
invalid_specs = {}
for column in ['Mileage', 'Engine', 'Power', 'Seats']:
    invalid_specs[column] = int(data[column].le(0).sum())
    data.loc[data[column].le(0), column] = np.nan
data.loc[data['Kilometers_Driven'].lt(0), 'Kilometers_Driven'] = np.nan
assert data['Year'].dropna().between(1900, REFERENCE_YEAR).all()
removed_targets = int((data['Price'].isna() | data['Price'].le(0)).sum())
data = data.loc[data['Price'].gt(0)].copy()
data['Car_Age'] = REFERENCE_YEAR - data['Year']
numeric = ['Car_Age', 'Kilometers_Driven', 'Mileage', 'Engine', 'Power', 'Seats']
categorical = ['Fuel_Type', 'Transmission', 'Owner_Type']
X, y = data[numeric + categorical], data['Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
assert set(X_train.index).isdisjoint(X_test.index)
print('Nonpositive specifications treated as missing:', invalid_specs)
print('Invalid targets excluded:', removed_targets)
print('Training rows:', len(X_train), '| Test rows:', len(X_test))

Nonpositive specifications treated as missing: {'Mileage': 68, 'Engine': 0, 'Power': 0, 'Seats': 1}
Invalid targets excluded: 0
Training rows: 4813 | Test rows: 1204


## Exploratory analysis
Explore the training partition so the held-out sample does not guide model design. Associations and group averages do not establish causal effects or customer willingness to pay.

In [3]:
training = X_train.assign(Price=y_train)
correlations = training[numeric + ['Price']].corr()['Price'].drop('Price')
print('Training correlations with Price:')
print(correlations.round(3).to_string())
print('Training price median by transmission:')
print(training.groupby('Transmission')['Price'].agg(['count', 'median']).round(2).to_string())
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(y_train, bins=40, color='#276e90')
axes[0].set(xlabel='Recorded price (source units)', ylabel='Training observations', title='Training price distribution')
axes[1].bar(correlations.index, correlations.values, color='#276e90')
axes[1].axhline(0, color='gray', linewidth=0.7)
axes[1].tick_params(axis='x', rotation=60)
axes[1].set(ylabel='Pearson correlation', title='Training associations with price')
fig.tight_layout()
fig.savefig(figures / 'training-exploration.svg')
plt.close(fig)

Training correlations with Price:
Car_Age             -0.302
Kilometers_Driven    0.005
Mileage             -0.334
Engine               0.661
Power                0.779
Seats                0.051
Training price median by transmission:
              count   median
Transmission                
Automatic      1377  16250.0
Manual         3436   4565.0


## Portfolio Extension: Regression Modeling
Compare a training-mean baseline, ordinary linear regression and a random forest. The forest allows nonlinear relationships and interactions. These specifications are fixed before evaluating the test set; no test-driven tuning or feature selection is performed. Numeric medians and categorical imputation/encoding are fit within each pipeline on training data only. Unseen categories are safely ignored.

This is one random holdout, not a temporal or external validation. More robust deployment evaluation would require collection dates, vehicle identity and another market sample.

In [4]:
def preprocessing():
    return ColumnTransformer([
        ('numeric', SimpleImputer(strategy='median'), numeric),
        ('categorical', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('encode', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))
        ]), categorical)
    ])

models = {
    'Training-mean baseline': DummyRegressor(strategy='mean'),
    'Linear regression': LinearRegression(),
    'Random forest': RandomForestRegressor(n_estimators=200, min_samples_leaf=3,
                                            random_state=SEED, n_jobs=1)
}
results, predictions = [], {}
for name, estimator in models.items():
    pipeline = Pipeline([('prepare', preprocessing()), ('model', estimator)])
    pipeline.fit(X_train, y_train)
    predicted = pipeline.predict(X_test)
    assert len(predicted) == len(y_test) and np.isfinite(predicted).all()
    predictions[name] = predicted
    mse = mean_squared_error(y_test, predicted)
    results.append({'Model': name, 'R2': r2_score(y_test, predicted),
                    'MAE': mean_absolute_error(y_test, predicted), 'MSE': mse,
                    'RMSE': np.sqrt(mse), 'Negative predictions': int((predicted < 0).sum())})
metrics = pd.DataFrame(results).set_index('Model')
print(metrics.round(4).to_string())
metrics.to_csv('model-metrics.csv')
summary = {'original_rows': len(raw), 'rows_after_cleaning': len(data),
           'train_rows': len(X_train), 'test_rows': len(X_test), 'random_state': SEED,
           'age_reference_year': REFERENCE_YEAR, 'invalid_specifications': invalid_specs,
           'models': results}
Path('results.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')

                            R2        MAE           MSE        RMSE  Negative predictions
Model                                                                                    
Training-mean baseline -0.0001  7230.0334  1.405518e+08  11855.4523                     0
Linear regression       0.6495  3847.3271  4.925688e+07   7018.3245                   128
Random forest           0.8329  1783.5548  2.347681e+07   4845.2871                     0


## Actual versus predicted comparison
The comparison below aggregates holdout records into actual-price deciles. It avoids publishing individual source records. Good agreement in a decile does not mean every prediction in that group is accurate. Error metrics above are calculated on individual holdout observations before aggregation.

In [5]:
comparison = pd.DataFrame({'actual': y_test, 'predicted': predictions['Random forest']})
comparison['price_group'] = pd.qcut(comparison['actual'], 10, duplicates='drop')
grouped = comparison.groupby('price_group', observed=True).agg(
    observations=('actual', 'size'), actual_mean=('actual', 'mean'),
    predicted_mean=('predicted', 'mean'))
grouped = grouped.reset_index(drop=True).rename_axis('Price decile (low to high)')
grouped.index += 1
print(grouped.round(2).to_string())
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(grouped['actual_mean'], grouped['predicted_mean'], 'o-', color='#276e90', label='Holdout decile means')
limits = [0, max(grouped['actual_mean'].max(), grouped['predicted_mean'].max()) * 1.05]
ax.plot(limits, limits, '--', color='gray', label='Perfect agreement')
ax.set(xlabel='Mean actual price (source units)', ylabel='Mean predicted price (source units)',
       title='Random forest: actual vs predicted')
ax.legend()
fig.tight_layout()
fig.savefig(figures / 'actual-vs-predicted.svg')
plt.close(fig)

                            observations  actual_mean  predicted_mean
Price decile (low to high)                                           
1                                    126      1561.98         2068.12
2                                    118      2683.73         3149.11
3                                    123      3457.64         3766.24
4                                    116      4201.03         4491.05
5                                    130      5075.31         5433.45
6                                    112      6020.00         6125.12
7                                    118      7337.46         7395.87
8                                    120      9859.33        10553.46
9                                    121     16120.66        16783.32
10                                   120     37810.75        32831.21


## Business interpretation and limitations
Use the baseline comparison to judge whether vehicle features add predictive information in this sample. Review errors across price ranges before using any estimate to support pricing. A nonlinear model can capture patterns but does not prove that changing a vehicle characteristic causes a price change.

Recommendations: verify currency, source rights and collection period; add make/model, condition, location and service-history information; validate on newer listings and inspect high-price errors before operational use. No revenue improvement or deployment is claimed. Missing specifications, extreme kilometer values, possible unidentified repeated vehicles and unknown sampling practices limit generalization. The dataset has no transaction/listing distinction established. The model is a portfolio demonstration, not a current market quote.